In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from math import sqrt, erfc



In [ ]:
FIG = Path("../figures")
FIG.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 12



In [ ]:
# ======================
# 1. ПАРАМЕТРЫ СИМУЛЯЦИИ
# ======================
nA = nB = 8000
pA_true = 0.12
pB_true = 0.135  # ожидаемый uplift
rng = np.random.default_rng(RANDOM_SEED)

print("nA =", nA, "nB =", nB)
print("true conv A/B:", pA_true, pB_true)
print("true lift (pp):", round((pB_true - pA_true) * 100, 3))



In [ ]:
# ======================
# 2. ГЕНЕРАЦИЯ КОНВЕРСИЙ
# ======================
A = rng.binomial(1, pA_true, nA)
B = rng.binomial(1, pB_true, nB)

pA, pB = A.mean(), B.mean()
print("observed convA:", round(pA, 4))
print("observed convB:", round(pB, 4))
print("observed lift_pp:", round((pB - pA) * 100, 3))



In [ ]:
# Таблица результатов
summary = pd.DataFrame({
    "group": ["A", "B"],
    "n": [nA, nB],
    "conversions": [A.sum(), B.sum()],
    "conversion_rate": [pA, pB],
})
summary



In [ ]:
# Столбчатая диаграмма конверсий A vs B
plt.figure(figsize=(7, 5))
sns.barplot(
    x=summary["group"],
    y=summary["conversion_rate"],
    hue=summary["group"],
    palette="viridis",
    legend=False,
)
plt.title("Observed conversion rates", fontsize=16, pad=16)
plt.ylabel("Conversion rate")
plt.ylim(0, max(summary["conversion_rate"]) * 1.35)
plt.grid(axis="y", linestyle="--", alpha=0.7)

# Подписи процентов над столбцами
for i, row in summary.iterrows():
    plt.text(
        i,
        row["conversion_rate"] + 0.003,
        f"{row['conversion_rate']:.2%}",
        ha="center",
        fontsize=12,
    )

plt.tight_layout()
plt.savefig(FIG / "ab_conversion_bars.png", dpi=120, bbox_inches="tight")
plt.show()



In [ ]:
# ======================
# 3. Z-TEST (one-sided)
# ======================
p_pool = (A.sum() + B.sum()) / (nA + nB)
se = sqrt(p_pool * (1 - p_pool) * (1 / nA + 1 / nB))
z = (pB - pA) / se
pval = 0.5 * erfc(z / sqrt(2))  # one-sided P(Z > z)

print("pooled p:", round(p_pool, 5))
print("SE:", round(se, 5))
print("z:", round(z, 3))
print("p_one_sided:", round(pval, 4))



In [ ]:
# Интерпретация
alpha = 0.05
significant = pval < alpha
print("alpha:", alpha)
print("significant uplift:", significant)



In [ ]:
# Доверительные интервалы (Wald) для наглядности
z95 = 1.96
se_A = sqrt(pA * (1 - pA) / nA)
se_B = sqrt(pB * (1 - pB) / nB)

ci = pd.DataFrame({
    "group": ["A", "B"],
    "rate": [pA, pB],
    "lo": [pA - z95 * se_A, pB - z95 * se_B],
    "hi": [pA + z95 * se_A, pB + z95 * se_B],
})
ci



In [ ]:
# Визуализация CI
plt.figure(figsize=(7, 5))
plt.errorbar(
    ci["group"],
    ci["rate"],
    yerr=[ci["rate"] - ci["lo"], ci["hi"] - ci["rate"]],
    fmt="o",
    capsize=8,
    color="#4c72b0",
    markersize=10,
)
plt.title("Conversion rates with 95% CI", fontsize=14, pad=12)
plt.ylabel("Conversion rate")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()



In [ ]:
# ======================
# 4. BOOTSTRAP РАСПРЕДЕЛЕНИЕ LIFT
# ======================
n_boot = 2000
lifts = []
for _ in range(n_boot):
    a = rng.choice(A, size=nA, replace=True).mean()
    b = rng.choice(B, size=nB, replace=True).mean()
    lifts.append((b - a) * 100)

lifts = np.array(lifts)
print("bootstrap lift mean pp:", round(lifts.mean(), 3))
print("bootstrap 95% CI:", round(np.percentile(lifts, 2.5), 3), "…", round(np.percentile(lifts, 97.5), 3))



In [ ]:
# Гистограмма bootstrap uplift
plt.figure(figsize=(8, 4))
sns.histplot(lifts, bins=40, color="#4c72b0", edgecolor="black")
plt.axvline(0, color="red", linestyle="--", label="no lift")
plt.axvline(lifts.mean(), color="green", linestyle="-", label="mean lift")
plt.title("Bootstrap distribution of lift (percentage points)", fontsize=13, pad=10)
plt.xlabel("Lift (pp)")
plt.legend()
plt.tight_layout()
plt.show()



In [ ]:
# Мощность / чувствительность: сколько симуляций ловят uplift
n_sims = 500
detected = 0
for _ in range(n_sims):
    a = rng.binomial(1, pA_true, nA)
    b = rng.binomial(1, pB_true, nB)
    pa, pb = a.mean(), b.mean()
    pp = (a.sum() + b.sum()) / (nA + nB)
    se_i = sqrt(pp * (1 - pp) * (1 / nA + 1 / nB))
    z_i = (pb - pa) / se_i
    p_i = 0.5 * erfc(z_i / sqrt(2))
    detected += int(p_i < alpha)

print("approx power:", round(detected / n_sims, 3))



In [ ]:
# Сводка метрик
result = {
    "conv_A": round(pA, 4),
    "conv_B": round(pB, 4),
    "lift_pp": round((pB - pA) * 100, 3),
    "z": round(z, 3),
    "p_one_sided": round(pval, 4),
    "significant@0.05": significant,
}
pd.Series(result)



In [ ]:
# Контрольный график: число конверсий
plt.figure(figsize=(7, 4))
sns.barplot(
    x=["A", "B"],
    y=[A.sum(), B.sum()],
    hue=["A", "B"],
    palette="viridis",
    legend=False,
)
plt.title("Raw conversion counts", fontsize=14, pad=12)
plt.ylabel("Conversions")
for i, v in enumerate([A.sum(), B.sum()]):
    plt.text(i, v + 20, str(int(v)), ha="center")
plt.tight_layout()
plt.show()

